<a href="https://colab.research.google.com/github/Nourin-Nusrat/CSE4261_DNN/blob/main/Assignment1/DNNAssignment1_densenet201_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import warnings
import sys
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

In [ ]:
import keras
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from keras import layers
import numpy as np

In [ ]:
(x_train_tmp, y_train_tmp), (x_test_tmp, y_test_tmp) = keras.datasets.cifar100.load_data()

train_filter = (y_train_tmp < 20).flatten()
test_filter = (y_test_tmp < 20).flatten()

x_train = x_train_tmp[train_filter]
y_train = y_train_tmp[train_filter]
x_test = x_test_tmp[test_filter]
y_test = y_test_tmp[test_filter]

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

trainY = to_categorical(y_train, num_classes=20)
testY = to_categorical(y_test, num_classes=20)

169001437/169001437 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [ ]:
input_shape = (32, 32, 3)
densenet201_model = keras.applications.DenseNet201(
    include_top=False,
    weights="imagenet",
    input_shape=input_shape,
    pooling='avg'
)
model = keras.Sequential(
    [
        keras.Input(shape=(32, 32, 3)),
        densenet201_model,

        layers.Flatten(),

        layers.Dropout(0.4),
        layers.Dense(512),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Dropout(0.4),
        layers.Dense(127),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Dense(20, activation='softmax')
    ]
)
model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)                │ (None, 2048)                │      23,587,712 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 2048)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 2048)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 512)                 │       1,049,088 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 512)                 │           2,048 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation (Activation)              │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 127)                 │          65,151 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 127)                 │             508 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_1 (Activation)            │ (None, 127)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 20)                  │           2,560 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 24,707,067 (94.25 MB)

 Trainable params: 24,652,669 (94.04 MB)

 Non-trainable params: 54,398 (212.49 KB)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)

epochs = 10
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss=keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=['accuracy',],
)

model.fit(x_train, trainY, epochs=epochs, callbacks=[early_stopping], validation_split=0.1)
model.save('densenet201.keras')

Epoch 1/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 96s 129ms/step - accuracy: 0.2071 - loss: 2.7083 - val_accuracy: 0.0570 - val_loss: 4.1951
Epoch 2/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 31s 31ms/step - accuracy: 0.5228 - loss: 1.6458 - val_accuracy: 0.1320 - val_loss: 3.3455
Epoch 3/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 10s 31ms/step - accuracy: 0.6210 - loss: 1.2902 - val_accuracy: 0.2790 - val_loss: 3.0055
Epoch 4/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 11s 32ms/step - accuracy: 0.6866 - loss: 1.0438 - val_accuracy: 0.5250 - val_loss: 1.8498
Epoch 5/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 11s 33ms/step - accuracy: 0.7341 - loss: 0.8947 - val_accuracy: 0.6050 - val_loss: 1.3958
Epoch 6/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 9s 33ms/step - accuracy: 0.7777 - loss: 0.7460 - val_accuracy: 0.5760 - val_loss: 1.6549
Epoch 7/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 9s 32ms/step - accuracy: 0.7992 - loss: 0.6677 - val_accuracy: 0.6160 - val_loss: 1.4598
Epoch 8/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 10s 30ms/step - accuracy: 0.8281 - loss: 0.5907 - v

In [ ]:
model.evaluate(x_test, testY)

63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 47ms/step - accuracy: 0.6394 - loss: 1.3871


[1.3986350297927856, 0.6420000195503235]